# VISIONPORT: MODEL ARTEFACT LIFECYCLE
### Google Colab practical using NVIDIA GPU, ONNX and TensorRT

**Session Outcome:** Load a YOLO model, inspect its PyTorch checkpoint, export it to ONNX, validate the ONNX graph, build an FP16 TensorRT engine, run GPU inference, compare performance and package the artefacts.

| PyTorch (`yolov8n.pt`) | ONNX (`yolov8n.onnx`) | TensorRT (`yolov8n.engine`) |
| :--- | :--- | :--- |
| Original checkpoint | Portable graph | GPU-optimized engine |

> **Important distinction:** ONNX is a portable model-exchange representation. A TensorRT engine is optimized for a particular NVIDIA GPU/software environment and should normally be rebuilt on the target deployment machine.

## 1. Configure Google Colab & Verify GPU / CUDA
Open Google Colab, select **Runtime > Change runtime type > T4 GPU** (or other NVIDIA GPU), and run the check below.

In [ ]:
# Cell 1 – Verify GPU and CUDA
import torch
import subprocess

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version used by PyTorch:", torch.version.cuda)
else:
    raise RuntimeError(
        "GPU is not enabled. Select Runtime > Change runtime type > GPU."
    )

# Display NVIDIA driver and GPU information
!nvidia-smi

## 2. Install Required Python Packages
Install `ultralytics` for YOLO, `onnx` for graph validation, and `onnxruntime` for inference testing.

In [ ]:
# Cell 2 – Install Ultralytics and ONNX packages
!pip install -q -U ultralytics onnx onnxruntime

In [ ]:
# Cell 3 – Check installed package versions
import ultralytics
import onnx
import onnxruntime as ort

print("Ultralytics:", ultralytics.__version__)
print("ONNX:", onnx.__version__)
print("ONNX Runtime:", ort.__version__)
print("ONNX Runtime providers:", ort.get_available_providers())

## 3. Load the YOLO Model & Inspect Parameters
Load pretrained `yolov8n.pt` and check its structure and parameter count.

In [ ]:
# Cell 4 – Load the checkpoint
from ultralytics import YOLO
from pathlib import Path
import torch

MODEL_PATH = "yolov8n.pt"
# Downloads the pretrained checkpoint if it is not already present.
model = YOLO(MODEL_PATH)

print("Model loaded.")
print("Task:", model.task)
print("Number of classes:", len(model.names))
print("Class names:", model.names)
print("Checkpoint exists:", Path(MODEL_PATH).exists())

In [ ]:
# Cell 5 – Inspect the model parameters
total_params = sum(
    parameter.numel()
    for parameter in model.model.parameters()
)
trainable_params = sum(
    parameter.numel()
    for parameter in model.model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)

## 4. Run a Baseline Prediction on the GPU
Run inference on a sample image using PyTorch CUDA backend.

In [ ]:
# Cell 6 – Run YOLO inference
from IPython.display import display
from PIL import Image
import requests
from io import BytesIO

IMAGE_URL = "https://ultralytics.com/images/bus.jpg"
response = requests.get(IMAGE_URL, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert("RGB")

results = model.predict(
    source=image,
    imgsz=640,
    device=0,  # NVIDIA GPU 0
    conf=0.25,
    verbose=False
)

print("Number of detections:", len(results[0].boxes))
display(Image.fromarray(results[0].plot()[..., ::-1]))

## 5. Export PyTorch to ONNX & Validate Graph
Export the YOLO PyTorch model to an ONNX graph representation with opset 12.

In [ ]:
# Cell 7 – Export to ONNX
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True,
    device=0
)

print("ONNX export completed.")
print("ONNX file:", onnx_path)

In [ ]:
# Cell 8 – Validate and inspect the ONNX graph
import onnx

onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)

print("ONNX checker: model is valid")
print("IR version:", onnx_model.ir_version)
print("Opset version:", onnx_model.opset_import[0].version)

print("\nInputs:")
for tensor in onnx_model.graph.input:
    dims = [
        d.dim_value if d.dim_value else d.dim_param
        for d in tensor.type.tensor_type.shape.dim
    ]
    print(tensor.name, dims)

print("\nOutputs:")
for tensor in onnx_model.graph.output:
    dims = [
        d.dim_value if d.dim_value else d.dim_param
        for d in tensor.type.tensor_type.shape.dim
    ]
    print(tensor.name, dims)

print("\nNumber of graph nodes:", len(onnx_model.graph.node))

## 6. Run a Simple ONNX Runtime Test
Perform a smoke test with random tensor `(1, 3, 640, 640)` on ONNX Runtime.

In [ ]:
# Cell 9 – Execute the ONNX graph with a dummy input
import numpy as np
import onnxruntime as ort

onnx_session = ort.InferenceSession(
    str(onnx_path),
    providers=["CPUExecutionProvider"]
)

input_info = onnx_session.get_inputs()[0]
print("Input name:", input_info.name)
print("Input shape:", input_info.shape)
print("Input type:", input_info.type)

# This fixed-size example matches a typical YOLO input: NCHW, 640 x 640.
dummy_input = np.random.rand(1, 3, 640, 640).astype(np.float32)
onnx_outputs = onnx_session.run(
    None,
    {input_info.name: dummy_input}
)

print("ONNX Runtime execution completed.")
for index, output in enumerate(onnx_outputs):
    print("Output", index, "shape:", output.shape)

## 7. Install and Verify TensorRT
Install the TensorRT Python package and verify import.

In [ ]:
# Cell 10 – Install TensorRT Python package
!pip install -q tensorrt

In [ ]:
# Cell 11 – Verify TensorRT import and version
import tensorrt as trt
import torch

print("TensorRT version:", trt.__version__)
print("CUDA available to PyTorch:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## 8. Build the TensorRT FP16 Engine from ONNX
Build the TensorRT engine from `yolov8n.onnx` with FP16 half precision.

In [ ]:
# Cell 12 – Export ONNX model to TensorRT engine
onnx_yolo = YOLO(str(onnx_path))

engine_path = onnx_yolo.export(
    format="engine",
    imgsz=640,
    half=True,       # FP16 precision
    device=0,        # First NVIDIA GPU
    workspace=4      # Workspace limit in GB, where supported
)

print("TensorRT engine export completed.")
print("Engine file:", engine_path)

## 9. Load the TensorRT Engine and Run Inference

In [ ]:
# Cell 13 – Load engine and predict
trt_model = YOLO(str(engine_path))

trt_results = trt_model.predict(
    source=image,
    imgsz=640,
    device=0,
    conf=0.25,
    verbose=False
)

print("TensorRT inference completed.")
print("Number of detections:", len(trt_results[0].boxes))
display(Image.fromarray(trt_results[0].plot()[..., ::-1]))

## 10. Compare PyTorch and TensorRT Inference Time & FPS
Benchmark average latency (ms) and throughput (FPS) over 30 runs with 5 warmups.

In [ ]:
# Cell 14 – Benchmark both models
import time
import torch

def benchmark(yolo_model, model_name, runs=30, warmup=5):
    # Warm-up helps initialize kernels and runtime resources.
    for _ in range(warmup):
        yolo_model.predict(
            source=image,
            imgsz=640,
            device=0,
            verbose=False
        )
    torch.cuda.synchronize()
    start_time = time.perf_counter()
    for _ in range(runs):
        yolo_model.predict(
            source=image,
            imgsz=640,
            device=0,
            verbose=False
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start_time
    average_seconds = elapsed / runs
    fps = 1.0 / average_seconds
    print("\n", model_name)
    print("Average end-to-end time (ms):", round(average_seconds * 1000, 2))
    print("Approximate calls per second:", round(fps, 2))
    return average_seconds, fps

pt_seconds, pt_fps = benchmark(model, "PyTorch model")
trt_seconds, trt_fps = benchmark(trt_model, "TensorRT FP16 engine")

print("\nComparison")
print("PyTorch FPS:", round(pt_fps, 2))
print("TensorRT FPS:", round(trt_fps, 2))
print("Measured speedup:", round(pt_seconds / trt_seconds, 2), "x")

## 11. Inspect and Package the Model Artefacts
Report file sizes and package `.pt`, `.onnx`, and `.engine` into `VisionPort_Artifacts.zip`.

In [ ]:
# Cell 15 – Report file sizes
from pathlib import Path

artifact_paths = [
    Path(MODEL_PATH),
    Path(onnx_path),
    Path(engine_path)
]

print("VISIONPORT – ARTEFACT REPORT")
print("-" * 48)
for path in artifact_paths:
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.name:25s} {size_mb:8.2f} MB")
    else:
        print("Missing:", path)

In [ ]:
# Cell 16 – Create a ZIP containing the artefacts
import zipfile
from pathlib import Path

zip_name = "VisionPort_Artifacts.zip"
with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as zip_file:
    for path in artifact_paths:
        if path.exists():
            zip_file.write(path, arcname=path.name)

print("Created:", zip_name)

In [ ]:
# Cell 17 – Download the ZIP to your computer
from google.colab import files
files.download("VisionPort_Artifacts.zip")

## 12. Use the Actual VisionPort Trained Checkpoint (Optional)
For the final VisionPort project, mount Google Drive and run with `best.pt`.

In [ ]:
# Optional cell – Load best.pt from Google Drive
from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")
MODEL_PATH = "/content/drive/MyDrive/VisionPort/best.pt"
model = YOLO(MODEL_PATH)
print("Loaded VisionPort checkpoint:", MODEL_PATH)
print("Task:", model.task)
print("Classes:", model.names)